# October Gate 1 - Split Design (`OCT-SPLIT-001`)

This notebook freezes the claim-level partition used for every October model comparison:
70% train / 15% validation / 15% test, random seed 42, stratified on 20 quantile bins of `loss`,
with 5 cross-validation folds inside train. The design and its reasons are recorded in
`docs/team/october/Gate 1/split_design_decision.md`.

The notebook writes its evidence next to itself and fills the `split` block of
`modeling_run_manifest.json`. It never computes a `loss` statistic or score for the test partition.

**Colab:** mount Drive and set `ROOT` in the next cell to the folder that contains
`data/allstate_claims_data.csv`.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "data/allstate_claims_data.csv").exists():
            return path
    raise FileNotFoundError(
        "Could not find data/allstate_claims_data.csv above the working directory. "
        "Set ROOT to the repository folder by hand."
    )


ROOT = find_repo_root(Path.cwd().resolve())
SOURCE_PATH = ROOT / "data/allstate_claims_data.csv"
OUT_DIR = ROOT / "notebooks/final-deliverables/October/Gate 1"
MANIFEST_PATH = OUT_DIR / "partition_manifest.csv"
RUN_MANIFEST_PATH = OUT_DIR / "modeling_run_manifest.json"

EXPECTED_SOURCE_SHA256 = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
SPLIT_VERSION = "OCT-SPLIT-001"
RANDOM_SEED = 42
TEST_SHARE = 0.15
VALIDATION_SHARE = 0.15
TRAIN_SHARE = 1 - TEST_SHARE - VALIDATION_SHARE
N_LOSS_BINS = 20
N_CV_FOLDS = 5

# Proposed thresholds, pending team review (see split_design_decision.md).
# Exceeding one records a flag for the team; it never redraws the split.
MAX_CENTER_DIFF = 0.02  # train vs validation mean and median
MAX_P99_DIFF = 0.05     # train vs validation p99

## Load the source and confirm its identity

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_sha256 = sha256_file(SOURCE_PATH)
df = pd.read_csv(SOURCE_PATH)
print(f"Source rows: {len(df):,}  columns: {df.shape[1]}")
print(f"Source SHA-256 matches contract: {source_sha256 == EXPECTED_SOURCE_SHA256}")

Source rows: 188,318  columns: 132
Source SHA-256 matches contract: True


## Build the split

Each claim gets a `loss` bin (20 quantile bins of 5% each). The split is drawn in two steps so
all three partitions keep the same mix of bins: first 15% of claims go to test, then 15/85 of the
remaining claims go to validation. Train claims are then divided into 5 stratified folds for
cross-validation. The bins only decide partition membership; they are not model inputs.

In [3]:
def build_manifest(ids, loss, seed):
    """Assign every claim to train/validation/test and every train claim to a CV fold."""
    ids = np.asarray(ids)
    loss_bin = pd.qcut(np.asarray(loss), q=N_LOSS_BINS, labels=False)
    positions = np.arange(len(ids))

    rest, test = train_test_split(
        positions, test_size=TEST_SHARE, stratify=loss_bin, random_state=seed
    )
    train, validation = train_test_split(
        rest,
        test_size=VALIDATION_SHARE / (1 - TEST_SHARE),
        stratify=loss_bin[rest],
        random_state=seed,
    )

    partition = np.empty(len(ids), dtype=object)
    partition[train] = "train"
    partition[validation] = "validation"
    partition[test] = "test"

    cv_fold = pd.array([pd.NA] * len(ids), dtype="Int64")
    folds = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=seed)
    for fold, (_, in_fold) in enumerate(folds.split(np.zeros(len(train)), loss_bin[train])):
        cv_fold[train[in_fold]] = fold

    manifest = pd.DataFrame({"id": ids, "partition": partition, "cv_fold": cv_fold})
    return manifest.sort_values("id", ignore_index=True)


def manifest_bytes(manifest):
    return manifest.to_csv(index=False, lineterminator="\n").encode("utf-8")


manifest = build_manifest(df["id"], df["loss"], RANDOM_SEED)
data = manifest_bytes(manifest)
MANIFEST_PATH.write_bytes(data)
manifest_sha256 = hashlib.sha256(data).hexdigest()
print(manifest["partition"].value_counts())
print(f"Manifest SHA-256: {manifest_sha256}")

partition
train         131822
test           28248
validation     28248
Name: count, dtype: int64
Manifest SHA-256: de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b


## Integrity checks

Each check is recorded with its expected and observed value. Any failed integrity check stops the notebook.

In [4]:
checks = []


def record(check_id, kind, description, expected, observed, passed):
    failed_status = "fail" if kind == "integrity" else "flag"
    checks.append({
        "check_id": check_id,
        "kind": kind,
        "description": description,
        "expected": str(expected),
        "observed": str(observed),
        "status": "pass" if passed else failed_status,
    })


n = len(df)
record("I1", "integrity", "Source SHA-256 matches the modeling contract",
       EXPECTED_SOURCE_SHA256, source_sha256, source_sha256 == EXPECTED_SOURCE_SHA256)

record("I2", "integrity", "id is unique in the source",
       n, df["id"].nunique(), df["id"].is_unique)

same_ids = set(manifest["id"]) == set(df["id"])
covered = len(manifest) == n and manifest["id"].is_unique and same_ids
record("I3", "integrity", "Every source id appears exactly once in the manifest",
       f"{n} unique ids matching source",
       f"{len(manifest)} rows, {manifest['id'].nunique()} unique, same set: {same_ids}",
       covered)

counts = manifest["partition"].value_counts()
shares = {"train": TRAIN_SHARE, "validation": VALIDATION_SHARE, "test": TEST_SHARE}
counts_ok = set(counts.index) == set(shares) and all(
    abs(counts[name] - share * n) <= 1 for name, share in shares.items()
)
record("I4", "integrity", "Partition counts match 70/15/15 within 1 claim",
       {name: round(share * n, 1) for name, share in shares.items()},
       {k: int(v) for k, v in counts.items()}, counts_ok)

train_folds = manifest.loc[manifest["partition"] == "train", "cv_fold"]
held_out_folds = manifest.loc[manifest["partition"] != "train", "cv_fold"]
fold_sizes = train_folds.value_counts().sort_index()
folds_ok = (
    not train_folds.isna().any()
    and set(fold_sizes.index) == set(range(N_CV_FOLDS))
    and fold_sizes.max() - fold_sizes.min() <= 0.01 * len(train_folds)
    and held_out_folds.isna().all()
)
record("I5", "integrity",
       "Each train claim has one fold 0-4, fold sizes within 1% of train, no folds outside train",
       "folds 0-4 on train only", {int(k): int(v) for k, v in fold_sizes.items()}, folds_ok)

rerun_sha256 = hashlib.sha256(
    manifest_bytes(build_manifest(df["id"], df["loss"], RANDOM_SEED))
).hexdigest()
record("I6", "integrity", "Regenerating with seed 42 gives an identical manifest",
       manifest_sha256, rerun_sha256, rerun_sha256 == manifest_sha256)

pd.DataFrame(checks)

,check_id,kind,description,expected,observed,status
0,I1,integrity,Source SHA-256 matches the modeling contract,74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf...,74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf...,pass
1,I2,integrity,id is unique in the source,188318,188318,pass
2,I3,integrity,Every source id appears exactly once in the ma...,188318 unique ids matching source,"188318 rows, 188318 unique, same set: True",pass
3,I4,integrity,Partition counts match 70/15/15 within 1 claim,"{'train': 131822.6, 'validation': 28247.7, 'te...","{'train': 131822, 'test': 28248, 'validation':...",pass
4,I5,integrity,"Each train claim has one fold 0-4, fold sizes ...",folds 0-4 on train only,"{0: 26365, 1: 26365, 2: 26364, 3: 26364, 4: 26...",pass
5,I6,integrity,Regenerating with seed 42 gives an identical m...,de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f...,de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f...,pass


In [5]:
failed = [c["check_id"] for c in checks if c["status"] == "fail"]
assert not failed, f"Integrity checks failed: {failed}"

## Save the checks and register the split

In [6]:
pd.DataFrame(checks).to_csv(OUT_DIR / "split_checks.csv", index=False)

run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding="utf-8"))
run_manifest["split"] = {
    "status": "frozen",
    "version": SPLIT_VERSION,
    "method": (
        "Stratified random split on 20 quantile bins of loss: 15% test drawn first, "
        "then 15/85 of the remainder as validation; 5-fold StratifiedKFold inside train"
    ),
    "random_seed": RANDOM_SEED,
    "train_claims": int(counts["train"]),
    "validation_claims": int(counts["validation"]),
    "test_claims": int(counts["test"]),
    "cv_folds": N_CV_FOLDS,
    "manifest_path": "notebooks/final-deliverables/October/Gate 1/partition_manifest.csv",
    "manifest_sha256": manifest_sha256,
    "decision_record": "docs/team/october/Gate 1/split_design_decision.md",
    "library_versions": {
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}
# The committed file uses CRLF line endings; keep them so the diff shows only the split block.
with RUN_MANIFEST_PATH.open("w", encoding="utf-8", newline="\r\n") as handle:
    handle.write(json.dumps(run_manifest, indent=2) + "\n")

print(json.dumps(run_manifest["split"], indent=2))

{
  "status": "frozen",
  "version": "OCT-SPLIT-001",
  "method": "Stratified random split on 20 quantile bins of loss: 15% test drawn first, then 15/85 of the remainder as validation; 5-fold StratifiedKFold inside train",
  "random_seed": 42,
  "train_claims": 131822,
  "validation_claims": 28248,
  "test_claims": 28248,
  "cv_folds": 5,
  "manifest_path": "notebooks/final-deliverables/October/Gate 1/partition_manifest.csv",
  "manifest_sha256": "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b",
  "decision_record": "docs/team/october/Gate 1/split_design_decision.md",
  "library_versions": {
    "scikit-learn": "1.6.1",
    "pandas": "2.3.3",
    "numpy": "2.0.2"
  }
}
